# A Gradio agent chatbot with safe traces

## Learning goals

- Keep agent runtime logic separate from the UI adapter
- Return a final answer and a small structured tool trace
- Preserve per-session chat history without global mutable state
- Convert internal errors into safe user-facing messages
- Build a Gradio UI that remains offline and testable by default


## Architecture before UI

The Gradio callback should not *be* the agent. It should validate the UI request, call an agent-facing function, and translate the result into components. This separation lets us test the runtime without a browser and replace the UI without rewriting agent logic.

This notebook uses an offline deterministic travel assistant. A later project can replace the drafting function with a live model call while keeping the same `TurnResult` contract.


## Mental model

```mermaid
flowchart LR
  A[Browser message plus session history] --> B[Gradio callback]
  B --> C[Input validation]
  C --> D[Agent runtime]
  D --> E[Authorized tool runtime]
  E --> D
  D --> F[TurnResult]
  F --> G[Final answer component]
  F --> H[Safe trace component]
  B -->|controlled failure| I[User-safe error]
```

The final answer is user-facing content. The trace is operational metadata: tool name, status, and high-level outcome. It is not chain-of-thought and must not include secrets, hidden prompts, authorization headers, or raw private records.


## Define the UI-facing runtime contract

Typed trace events keep the UI independent from internal exceptions and arbitrary debug strings. An allowlisted schema also makes accidental secret leakage less likely than returning entire runtime objects.


In [ ]:
from typing import Any, Literal

from pydantic import BaseModel, Field


class TraceEvent(BaseModel):
    step: int = Field(ge=1)
    event: Literal["input_validated", "tool_result", "final", "error"]
    tool: str | None = None
    status: Literal["ok", "skipped", "failed"]
    detail: str = Field(max_length=200)


class TurnResult(BaseModel):
    answer: str = Field(min_length=1, max_length=2_000)
    trace: list[TraceEvent]


## A small offline tool

The tool returns local teaching data and a source label. The runtime exposes only a match count in the trace; the final answer can use the matched note, but the UI does not receive the entire internal corpus as debug output.


In [ ]:
TRAVEL_NOTES = {
    "jaipur": [
        "Amber Fort is best treated as a half-day outing from central Jaipur.",
        "A relaxed old-city day can group City Palace, Jantar Mantar, and nearby markets.",
    ],
    "jodhpur": [
        "Mehrangarh Fort and the old city can anchor a relaxed first day.",
    ],
}


def search_travel_notes(query: str) -> list[str]:
    lowered = query.lower()
    matches = []
    for city, notes in TRAVEL_NOTES.items():
        if city in lowered:
            matches.extend(notes)
    return matches[:3]


## The pure agent-facing function

This runtime has one deterministic routing rule: use local notes when a supported city appears. The trace records observable events only. Browser history is accepted as input but is not treated as authenticated memory or permission state.


In [ ]:
MAX_MESSAGE_LENGTH = 1_000


def run_agent_turn(message: str, history: list[dict[str, Any]]) -> TurnResult:
    cleaned = (message or "").strip()
    if not cleaned:
        raise ValueError("Message cannot be blank.")
    if len(cleaned) > MAX_MESSAGE_LENGTH:
        raise ValueError(f"Message exceeds {MAX_MESSAGE_LENGTH} characters.")

    trace = [TraceEvent(
        step=1,
        event="input_validated",
        status="ok",
        detail=f"Accepted message; UI supplied {len(history)} prior message(s).",
    )]

    notes = search_travel_notes(cleaned)
    if notes:
        trace.append(TraceEvent(
            step=2,
            event="tool_result",
            tool="search_travel_notes",
            status="ok",
            detail=f"Found {len(notes)} local note(s).",
        ))
        answer = f"Here is a grounded starting point: {notes[0]}"
    else:
        trace.append(TraceEvent(
            step=2,
            event="tool_result",
            tool="search_travel_notes",
            status="skipped",
            detail="No supported city matched the local note collection.",
        ))
        answer = "I can currently ground answers for Jaipur or Jodhpur. Which would you like to explore?"

    trace.append(TraceEvent(
        step=3,
        event="final",
        status="ok",
        detail="Prepared a user-facing answer.",
    ))
    return TurnResult(answer=answer, trace=trace)


## Test runtime behavior without Gradio

Testing the pure function catches most application mistakes faster than clicking through a browser. We verify both a grounded response and a fallback.


In [ ]:
jaipur_result = run_agent_turn("Plan two relaxed days in Jaipur.", [])
fallback_result = run_agent_turn("Plan a trip to Kochi.", [])

assert jaipur_result.trace[1].status == "ok"
assert fallback_result.trace[1].status == "skipped"
print(jaipur_result.answer)
print([event.model_dump() for event in jaipur_result.trace])


## Adapt `TurnResult` to Gradio

`ChatInterface` expects the reply first. `additional_outputs` lets the same callback update a separate JSON trace component. The adapter catches expected validation errors separately from unexpected internal failures and never returns raw exception text for the latter.


In [ ]:
def chat_with_trace(
    message: str,
    history: list[dict[str, Any]],
) -> tuple[str, list[dict[str, Any]]]:
    try:
        result = run_agent_turn(message, history)
        return result.answer, [event.model_dump(mode="json") for event in result.trace]
    except ValueError as error:
        safe_trace = [{
            "step": 1,
            "event": "error",
            "tool": None,
            "status": "failed",
            "detail": str(error),
        }]
        return str(error), safe_trace
    except Exception:
        safe_trace = [{
            "step": 1,
            "event": "error",
            "tool": None,
            "status": "failed",
            "detail": "Internal error; inspect protected server logs.",
        }]
        return "Something went wrong. Please try again.", safe_trace


reply, trace = chat_with_trace("Jaipur ideas", [])
assert reply
assert trace[1]["tool"] == "search_travel_notes"
print(reply)


## Build the chat and trace layout

The trace view shows only the latest turn. Chat history is managed per browser session by `ChatInterface`; no process-wide `history = []` is used. A production deployment should add authenticated server-side persistence only when the product requires durable history.


In [ ]:
import gradio as gr


with gr.Blocks() as agent_demo:
    gr.Markdown("# Offline travel agent")
    gr.Markdown("Ask about Jaipur or Jodhpur. The trace shows safe runtime events, not private reasoning.")
    trace_output = gr.JSON(label="Latest turn trace")
    gr.ChatInterface(
        fn=chat_with_trace,
        chatbot=gr.Chatbot(
            label="Conversation",
            sanitize_html=True,
            height=420,
        ),
        additional_outputs=[trace_output],
        examples=[
            "Plan two relaxed days in Jaipur.",
            "What should I prioritize in Jodhpur?",
        ],
        api_visibility="private",
        concurrency_limit=4,
        save_history=False,
    )

print(type(agent_demo).__name__)


## Optional launch

The server is disabled during automated validation. Keep `share=False` for local course work containing private prompts or credentials.


In [ ]:
RUN_GRADIO = False

if RUN_GRADIO:
    agent_demo.launch(
        inline=True,
        share=False,
        show_error=False,
    )
else:
    print("Agent UI not launched. Set RUN_GRADIO = True when ready.")


## Production checklist

- Authenticate before loading user-specific history or tools.
- Keep session state isolated; never use one module-level list for all users.
- Bound input, output, steps, tool calls, time, concurrency, and cost.
- Require explicit confirmation for consequential tool actions.
- Cancel downstream model/tool work when the user stops a request where supported.
- Redact traces and logs; do not expose hidden prompts or chain-of-thought.
- Define retention and deletion policy for chat history and feedback.
- Test pure runtime logic, UI adapters, and browser behavior separately.
- Add health checks and safe fallbacks for provider outages.


## Exercise

1. Add a city dropdown and pass it as an additional input without trusting it for authorization.
2. Add a bounded step counter to `run_agent_turn`.
3. Add a simulated tool failure and verify the user sees a safe fallback.
4. Show tool duration in the trace without exposing tool arguments.
5. Replace the deterministic answer composer with the guarded OpenAI helper from notebook `01`, while keeping `RUN_LIVE_CALL=False` by default.


## Summary

A Gradio agent chatbot is a UI adapter around a bounded runtime. Keep the runtime pure and testable, return typed final answers plus allowlisted trace events, isolate session history, and translate failures into safe UI messages. A useful trace shows what the runtime did; it does not reveal private reasoning or secrets.
